<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Javier Pinochet
- Nombre de alumno 2: Daniel Muñoz


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/danmc899/MDS7202)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [ ]:
!pip install -qq xgboost optuna

# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive/')

df = pd.read_csv("/content/drive/MyDrive/sales.csv")

df.head()

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [ ]:
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
set_config(transform_output="pandas")

# Inserte su código acá

# 1.
RANDOM_STATE = 42

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=RANDOM_STATE)
val_df, test_df = train_test_split(temp_df, test_size=1/3, random_state=RANDOM_STATE)
print("PARTE 1.")
print(len(train_df), len(val_df), len(test_df))

# 2.
def extract_date_features(X):
    X = X.copy()
    X["date"] = pd.to_datetime(X["date"], format="%d/%m/%y")
    X["day"] = X["date"].dt.day.astype("category")
    X["month"] = X["date"].dt.month.astype("category")
    X["year"] = X["date"].dt.year.astype("category")
    return X.drop(columns=["date"])

date_transformer = FunctionTransformer(extract_date_features)

df_trans = date_transformer.fit_transform(df)
print("PARTE 2.")
print(df_trans.head())

# 3.
# Definir columnas
num_cols = ["lat", "long", "pop", "price"]
cat_cols = ["city", "shop", "brand", "container", "capacity", "day", "month", "year"]

# Definir transformaciones (nota el sparse_output=False)
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
]).set_output(transform="pandas")

pipeline = Pipeline([
    ("date_features", date_transformer),
    ("preprocessor", preprocessor)
])

X_train_trans = pipeline.fit_transform(train_df)
print("PARTE 3.")
print(X_train_trans.head())


PARTE 1.
5219 1491 746
PARTE 2.
   id    city       lat      long     pop    shop        brand container  \
0   0  Athens  37.97945  23.71622  672130  shop_1  kinder-cola     glass   
1   1  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   plastic   
2   2  Athens  37.97945  23.71622  672130  shop_1  kinder-cola       can   
3   3  Athens  37.97945  23.71622  672130  shop_1   adult-cola     glass   
4   4  Athens  37.97945  23.71622  672130  shop_1   adult-cola       can   

  capacity  price  quantity day month  year  
0    500ml   0.96     13280  31     1  2012  
1    1.5lt   2.86      6727  31     1  2012  
2    330ml   0.87      9848  31     1  2012  
3    500ml   1.00     20050  31     1  2012  
4    330ml   0.39     25696  31     1  2012  
PARTE 3.
      num__lat  num__long  num__pop  num__price  cat__city_Athens  \
292  -0.041439  -1.403508 -0.815231    1.659156               0.0   
3366 -0.201218   0.416377  1.362723   -0.594186               1.0   
3685 -0.211468   0.

In [ ]:
# 4.
# Variable objetivo
y_train = train_df["quantity"]
y_val = val_df["quantity"]

# Variables predictoras (quitamos la target)
X_train = train_df.drop(columns=["quantity"])
X_val = val_df.drop(columns=["quantity"])

# Pipeline completo
dummy_pipe = Pipeline([
    ("date_features", date_transformer),
    ("preprocessor", preprocessor),
    ("model", DummyRegressor(strategy="mean"))
])
print("PARTE 4.")

# 5.
# Entrenar
dummy_pipe.fit(X_train, y_train)

# Predecir en validación
y_pred = dummy_pipe.predict(X_val)

# Calcular MAE
mae_dummy = mean_absolute_error(y_val, y_pred)
print("PARTE 5.")
print("MAE DummyRegressor:", mae_dummy)

# 6.
from xgboost import XGBRegressor

# Pipeline con XGBoost
xgb_pipe = Pipeline([
    ("date_features", date_transformer),
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(random_state=RANDOM_STATE))
])

# Entrenar
xgb_pipe.fit(X_train, y_train)

# Predecir
y_pred_xgb = xgb_pipe.predict(X_val)

# Calcular MAE
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
print("PARTE 6.")
print("MAE XGBRegressor:", mae_xgb)

PARTE 4.
PARTE 5.
MAE DummyRegressor: 13298.497767341096
PARTE 6.
MAE XGBRegressor: 2433.321044921875


**¿Cómo se interpreta esta métrica para el contexto del negocio?**

El MAE es el error medio absoluto entre las predicciones y los valores reales observados, por tanto en el caso de DummyRegressor, indica que el modelo se aleja del valor observado real alrededor de unas 13.300 unidades de registro entre la suma absoluta de todos los errores.

**¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el DummyRegressor?**

XGBRegressor mejora notablemente el desempeño, bajando el MAE a 2.433, esto porque captura relaciones reales entre las variables. Por tanto es mucho mejor que el DummyRegressor



In [ ]:
import joblib
# 7.

# Guardar el pipeline completo (incluye preprocesamiento + modelo)
joblib.dump(dummy_pipe, "dummy_regressor.pkl")
joblib.dump(xgb_pipe, "xgb_regressor.pkl")

print("PARTE 7")
print("Modelos guardados correctamente.")

PARTE 7
Modelos guardados correctamente.


## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [ ]:
# Inserte su código acá
# 1.
# Ajustamos hasta antes del modelo
preprocessor_pipe = Pipeline([
    ("date_features", date_transformer),
    ("preprocessor", preprocessor)
])
X_train_proc = preprocessor_pipe.fit_transform(X_train)

# Nombres finales de las columnas
feature_names = preprocessor_pipe.named_steps["preprocessor"].get_feature_names_out()
print(feature_names)

['num__lat' 'num__long' 'num__pop' 'num__price' 'cat__city_Athens'
 'cat__city_Irakleion' 'cat__city_Larisa' 'cat__city_Patra'
 'cat__city_Thessaloniki' 'cat__shop_shop_1' 'cat__shop_shop_2'
 'cat__shop_shop_3' 'cat__shop_shop_4' 'cat__shop_shop_5'
 'cat__shop_shop_6' 'cat__brand_adult-cola' 'cat__brand_gazoza'
 'cat__brand_kinder-cola' 'cat__brand_lemon-boost'
 'cat__brand_orange-power' 'cat__container_can' 'cat__container_glass'
 'cat__container_plastic' 'cat__capacity_1.5lt' 'cat__capacity_330ml'
 'cat__capacity_500ml' 'cat__day_28' 'cat__day_29' 'cat__day_30'
 'cat__day_31' 'cat__month_1' 'cat__month_2' 'cat__month_3' 'cat__month_4'
 'cat__month_5' 'cat__month_6' 'cat__month_7' 'cat__month_8'
 'cat__month_9' 'cat__month_10' 'cat__month_11' 'cat__month_12'
 'cat__year_2012' 'cat__year_2013' 'cat__year_2014' 'cat__year_2015'
 'cat__year_2016' 'cat__year_2017' 'cat__year_2018']


In [ ]:
# 2.
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Crear diccionario de constraints
monotone_dict = {name: 0 for name in feature_names}
monotone_dict["num__price"] = -1  # inversa con la cantidad



xgb_mono_pipe = Pipeline([
    ("date_features", date_transformer),
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        random_state=RANDOM_STATE,
        monotone_constraints=monotone_dict
    ))
])

xgb_mono_pipe.fit(X_train, y_train)
y_pred_mono = xgb_mono_pipe.predict(X_val)
mae_mono = mean_absolute_error(y_val, y_pred_mono)
print(f"MAE XGB con restricción monótona: {mae_mono:.2f}")

MAE XGB con restricción monótona: 2485.27


3. El error MAE aumenta levemente a 2485, esto porque se impone que el precio tiene un comportamiento inverso sobre la demanda. Sin embargo, la afirmación de nuestro amigo es que el precio es inversamente proprocional a la demanda, no se refiere tanto al error, por lo tanto, tiene razón del punto de vista ecónomico que el precio es inverso a la demanda, pero a pesar de subir levemente el error, el modelo gana cierta coherencia e interpretabilidad al imponerse esta relación entre dos variables.

In [ ]:
#4.
import joblib

joblib.dump(xgb_mono_pipe, "xgb_regressor_monotono.pkl")
print("Modelo XGBRegressor con restricción monótona guardado correctamente.")

Modelo XGBRegressor con restricción monótona guardado correctamente.


## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [ ]:
import optuna
from optuna.samplers import TPESampler
import time

optuna.logging.set_verbosity(optuna.logging.WARNING)
# Inserte su código acá

def objective(trial):
    # Hiperparámetros para OneHotEncoder
    min_freq = trial.suggest_float("min_frequency", 0.0, 1.0)

    # Hiperparámetros para XGBRegressor
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "max_leaves": trial.suggest_int("max_leaves", 0, 100),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": RANDOM_STATE,
    }

    # Crear preprocesador con el hiperparámetro min_frequency
    preprocessor_optuna = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=min_freq), cat_cols)
    ], remainder='passthrough').set_output(transform="pandas")

    # Crear un pipeline de preprocesamiento para obtener los nombres de las características
    preprocessor_pipe_optuna = Pipeline([
        ("date_features", date_transformer),
        ("preprocessor", preprocessor_optuna)
    ])

    preprocessor_pipe_optuna.fit(X_train, y_train)
    feature_names_out = preprocessor_pipe_optuna.named_steps['preprocessor'].get_feature_names_out()

    # Crear el diccionario de restricciones
    monotone_dict_optuna = {name: 0 for name in feature_names_out}
    if "num__price" in monotone_dict_optuna:
        monotone_dict_optuna["num__price"] = -1

    params["monotone_constraints"] = monotone_dict_optuna

    # Recrear el pipeline completo con el modelo configurado
    pipeline_optuna = Pipeline([
        ("date_features", date_transformer),
        ("preprocessor", preprocessor_optuna),
        ("model", XGBRegressor(**params))
    ])

    # Entrenar el pipeline
    pipeline_optuna.fit(X_train, y_train)

    # Evaluación
    y_pred = pipeline_optuna.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)

    # Guardar el pipeline entrenado en el trial
    trial.set_user_attr("best_pipeline", pipeline_optuna)

    return mae

# Sampler
sampler = TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, timeout=300, show_progress_bar=True) # Timeout de 5 minutos

# Resultados
print(f"Número de trials completados: {len(study.trials)}")
print(f"Mejor MAE (validación): {study.best_value}")
print("Mejores hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

# Guardar
best_pipeline = study.best_trial.user_attrs["best_pipeline"]
joblib.dump(best_pipeline, "xgb_regressor_optuna.pkl")
print("\nMejor modelo guardado")

   0%|          | 00:00/05:00

Número de trials completados: 139
Mejor MAE (validación): 2077.0947265625
Mejores hiperparámetros encontrados:
  - min_frequency: 0.07947441135123287
  - learning_rate: 0.04375207817190302
  - n_estimators: 952
  - max_depth: 10
  - max_leaves: 98
  - min_child_weight: 4
  - reg_alpha: 0.10275682983969434
  - reg_lambda: 0.19692267759436682

Mejor modelo guardado


El MAE bajó de 2485 a 2077 al optimizar los hiperparámetros, esto se debe a que se encontraron mejores combinaciones de parámetros que permiten al modelo ajustarse mejor a los datos y capturar patrones más complejos en la relación entre las características y la variable objetivo.

### Hiperparámetros de `XGBRegressor`

*   **`learning_rate`**:
    *   **Función**: Controla el peso que se le da a cada nuevo árbol añadido al modelo. Valores más bajos hacen el aprendizaje más lento pero más robusto.
    *   **Rango (0.001, 0.1)**: Es un rango estándar y efectivo. Valores más bajos previenen el sobreajuste, mientras que valores más altos aceleran el entrenamiento. La escala logarítmica es ideal para explorar este parámetro.

*   **`n_estimators`**:
    *   **Función**: Es el número total de árboles que se construirán en el ensamblaje.
    *   **Rango (50, 1000)**: Ofrece un buen balance. Menos de 50 podría subajustar, y más de 1000 puede sobreajustar y aumentar mucho el tiempo de entrenamiento.

*   **`max_depth`**:
    *   **Función**: Define la profundidad máxima de cada árbol. Limita la complejidad del modelo.
    *   **Rango (3, 10)**: Es un rango común. Profundidades bajas (3) crean modelos simples, mientras que profundidades altas (10) permiten capturar interacciones complejas, con riesgo de sobreajuste.

*   **`max_leaves`**:
    *   **Función**: Número máximo de nodos hoja en un árbol. Un valor de 0 significa sin límite. Es una forma alternativa a `max_depth` para controlar la complejidad.
    *   **Rango (0, 100)**: Permite explorar desde árboles sin límite de hojas (enfocados en la profundidad) hasta árboles con un número controlado de hojas, lo que puede generar modelos más generalizables.

*   **`min_child_weight`**:
    *   **Función**: Suma mínima de pesos de instancia necesaria en una hoja. Ayuda a prevenir el sobreajuste al evitar que el modelo aprenda de grupos de datos muy pequeños.
    *   **Rango (1, 5)**: Es un rango conservador y efectivo. El valor por defecto es 1. Aumentarlo hace al modelo más robusto.

*   **`reg_alpha` (L1)** y **`reg_lambda` (L2)**:
    *   **Función**: Términos de regularización que penalizan la complejidad del modelo para evitar el sobreajuste. `reg_alpha` puede llevar a pesos cero (selección de características), mientras que `reg_lambda` reduce los pesos.
    *   **Rango (1e-8, 1.0)**: Permite probar desde una regularización casi nula hasta una moderada. La escala logarítmica es útil para explorar eficientemente valores cercanos a cero.

### Hiperparámetro de `OneHotEncoder`

*   **`min_frequency`**:
    *   **Función**: Ignora las categorías que aparecen con una frecuencia relativa menor a este valor. Ayuda a reducir el ruido y la dimensionalidad causada por categorías muy raras.
    *   **Rango (0.0, 1.0)**: Permite explorar todo el espectro, desde incluir todas las categorías (0.0) hasta agrupar todas las categorías poco frecuentes en una sola columna de "infrecuentes".

## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

El **pruning** (o poda) es una técnica que detiene de forma temprana los *trials* (pruebas de hiperparámetros) que no son prometedores. En lugar de esperar a que un entrenamiento con una mala combinación de hiperparámetros termine, el pruner lo detiene a mitad de camino si su rendimiento intermedio es malo en comparación con otros trials.

Debería impactar el entrenamiento de la siguiente manera:

*   **Acelera la optimización**: Al descartar rápidamente las malas combinaciones de hiperparámetros, se ahorra tiempo de cómputo.
*   **Permite más trials**: En el mismo período de tiempo (por ejemplo, 5 minutos), se pueden explorar muchas más combinaciones de hiperparámetros.
*   **Mejora potencial del resultado**: Al poder probar más combinaciones, aumenta la probabilidad de encontrar un conjunto de hiperparámetros que resulte en un mejor modelo (menor MAE).

In [ ]:
!pip install optuna-integration[xgboost]

In [ ]:
!pip install --upgrade xgboost

In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.integration import XGBoostPruningCallback
import time
import xgboost
from xgboost.callback import EarlyStopping

optuna.logging.set_verbosity(optuna.logging.WARNING)
# Inserte su código acá

def objective(trial):
    # Definir hiperparámetros
    min_freq = trial.suggest_float("min_frequency", 0.0, 1.0)
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "max_leaves": trial.suggest_int("max_leaves", 0, 100),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "random_state": RANDOM_STATE,
        "eval_metric": "mae"
    }

    # Pipeline de preprocesamiento
    preprocessor_optuna = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=min_freq), cat_cols)
    ], remainder='passthrough').set_output(transform="pandas")

    preprocessing_pipeline = Pipeline([
        ("date_features", date_transformer),
        ("preprocessor", preprocessor_optuna)
    ])

    # Ajustar el preprocesador y transformar los datos
    X_train_transformed = preprocessing_pipeline.fit_transform(X_train, y_train)
    X_val_transformed = preprocessing_pipeline.transform(X_val)

    # Configurar el precio con demanda inversa
    feature_names_out = preprocessing_pipeline.named_steps['preprocessor'].get_feature_names_out()
    monotone_dict_optuna = {name: 0 for name in feature_names_out}
    if "num__price" in monotone_dict_optuna:
        monotone_dict_optuna["num__price"] = -1

    params["monotone_constraints"] = monotone_dict_optuna

    # Callback para el pruning
    pruning_callback = XGBoostPruningCallback(trial, "validation_0-mae")

    # Pasar el callback al constructor del modelo
    params["callbacks"] = [pruning_callback]

    model = XGBRegressor(**params)

    # Entrenar el modelo con los datos ya transformados
    # `eval_set` se pasa a .fit(), pero `callbacks` ya está en el constructor
    model.fit(
        X_train_transformed, y_train,
        eval_set=[(X_val_transformed, y_val)],
        verbose=False
    )

    # Reconstruir el pipeline completo para guardarlo
    full_pipeline = Pipeline([
        ("preprocessing", preprocessing_pipeline),
        ("model", model)
    ])

    trial.set_user_attr("best_pipeline", full_pipeline)

    y_pred = model.predict(X_val_transformed)
    mae = mean_absolute_error(y_val, y_pred)

    return mae

# Sampler
sampler = TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, timeout=300, show_progress_bar=True) # Timeout de 5 minutos

# Resultados
print(f"Número de trials completados: {len(study.trials)}")
print(f"Mejor MAE (validación): {study.best_value}")
print("Mejores hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

# Guardar
best_pipeline = study.best_trial.user_attrs["best_pipeline"]
joblib.dump(best_pipeline, "xgb_prunning_optuna.pkl")
print("\nMejor modelo guardado")

   0%|          | 00:00/05:00

Número de trials completados: 165
Mejor MAE (validación): 2137.3681640625
Mejores hiperparámetros encontrados:
  - min_frequency: 0.05609189488411281
  - learning_rate: 0.0911344005218463
  - n_estimators: 929
  - max_depth: 7
  - max_leaves: 76
  - min_child_weight: 5
  - reg_alpha: 0.000768611415794223
  - reg_lambda: 4.1053824547085546e-07

Mejor modelo guardado


**Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto?**

En nuestro caso, la optimización con prunning resultó en valores de MAE levemente más altos que en la sección anterior (2137 vs 2077), esto puedde deberse a que el prunning prioriza un poco la velocidad por sobre el rendimiento, con el fin de explorar una mayor cantidad de trials, eliminando configuraciones que no eran candidatas a optimizar el MAE, sin embargo, puede que la eliminación adelantada de estas configuraciones hayan eliminado ramas de combinaciones que podrían llevar al mismo o mejor resultado que el obtenido en la sección anterior. Entonces, probablemente este resultado con un poco menos de rendimiento que la sección anterior puede deberse a que con el afán de explorar más trials, el método de prunning eliminó configuraciones que de haber sido más exploradas podrían haber llevado a un mejor MAE óptimo.

## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [ ]:
# Inserte su código acá
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances

# 1.
plot_optimization_history(study)

In [ ]:
# 2.
plot_parallel_coordinate(study)

In [ ]:
# 3.
plot_param_importances(study)

4. Observando el gráfico de la historia de optimización, es posible ver que desde el trial 134 el MAE (en ese momento de 2151) llega a una meseta levemente descendiente, que a la larga, termina disminuyendo hasta su mejor versión (2137).

5. Observando el gráfico de coordenadas paralelas, hay hiperparámetros que no parecen tener efectos fuertes, tales como max_leaves, min_child_weight, n_estimators y reg_lambda.
También hay otros hiperparámetros que parecen tener una leve influencia en la elección del MAE óptimo, como max_depth (tiene una concentración de líneas en 7), reg_alpha (en 0.001) y reg_lambda (en 1e-6).
Por último, hay hiperparámetros que muestran una tendencia clara para minimizar el MAE, los cuales son learning_rate (tiene una concentración de líneas en 0.0999) y min_frequency (en 0.00022).

6. Observando el gráfico de importancia de hiperparámetros, el hiperparámetro más importante es claramente min_frequency con un 68%, seguido muy de lejos por learning_rate con un 13% y max_leaves con un 10%.

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [ ]:
# Inserte su código acá
# 1.
import pandas as pd

# Resultados obtenidos antes
mae_dummy = 13298.50
mae_xgb = 2433.32
mae_xgb_mono = 2485.27
mae_xgb_optuna = 2077.09
mae_xgb_optuna_prun = 2137.37

# Crear tabla resumen
mae_summary = pd.DataFrame({
    "Modelo": [
        "DummyRegressor (Baseline)",
        "XGBRegressor",
        "XGBRegressor con restricción monótona",
        "XGBRegressor con Optuna",
        "XGBRegressor con Optuna y prunners"
    ],
    "MAE (validación)": [
        mae_dummy,
        mae_xgb,
        mae_xgb_mono,
        mae_xgb_optuna,
        mae_xgb_optuna_prun
    ]
})

print(mae_summary)

                                  Modelo  MAE (validación)
0              DummyRegressor (Baseline)          13298.50
1                           XGBRegressor           2433.32
2  XGBRegressor con restricción monótona           2485.27
3                XGBRegressor con Optuna           2077.09
4     XGBRegressor con Optuna y prunners           2137.37


2. De la tabla anterior, el mejor modelo en cuanto a rendimiento es XGBRegressor con Optuna sin la implementación de prunners con un MAE de 2077, aunque seguido del XGBRegressor con Optuna con prunners con un MAE de 2137.

In [39]:
import joblib
from sklearn.metrics import mean_absolute_error

X_test = test_df.drop(columns=["quantity"])
y_test = test_df["quantity"]

# Cargar el mejor modelo
best_model = joblib.load("xgb_regressor_optuna.pkl")

# Predecir sobre el conjunto de test
y_pred_test = best_model.predict(X_test)

# Calcular MAE en test
mae_test = mean_absolute_error(y_test, y_pred_test)
print(f"MAE del mejor modelo (XGBRegressor) en test: {mae_test:.2f}")

MAE del mejor modelo (XGBRegressor) en test: 2088.34


4. La diferencia entre el MAE en el conjunto de validación (2077) y el de test (2088), es muy baja pero refleja un MAE levemente mayor para el conjunto de prueba, lo que puede traducirse en el hecho de que la optimización de hiperparámetros fue hecha sobre el conjunto de validación, por ello ajusta mejor este último, antes que al conjunto de prueba, que es un conjunto nuevo con el que se trabaja, y del cuál el modelo no tiene el mismo conocimiento. Aún así el MAE para el conjunto de test es un error aceptable dentro del rango esperado y refleja que es un modelo que puede generalizar bien en casos de prueba.

# Conclusión
Exito!
<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>